In [7]:
import pandas as pd
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)

In [17]:
METADATA_PATH = "s3://mateomorin/legifrance/metadata/acco_metadata_2025.parquet"

metadata = pd.read_parquet(METADATA_PATH, filesystem=fs)

theme_cols = [col for col in metadata.columns if col.startswith('theme_')]

# 2. Pivoter et créer la table d'appartenance
metadata_dummies = (
    pd.get_dummies(metadata[theme_cols].stack())
    .groupby(level=0)
    .max()
    .astype(bool)
    .reindex(metadata.index, fill_value=False)
)

metadata = metadata.drop(columns=theme_cols).join(metadata_dummies)

metadata.to_parquet(METADATA_PATH, filesystem=fs)